In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [2]:
# =====================================================================
# 1. RETURNS AND STALE PRICES
# =====================================================================

def compute_returns(prices_df):
    """
    Computes simple monthly returns: R_t = P_t / P_{t-1} - 1.
    Delisting is handled upstream.
    Returns are clipped at -100% to prevent theoretical negative prices.
    """
    returns = prices_df.pct_change(axis=1, fill_method=None)
    returns = returns.clip(lower=-1.0)
    return returns

def get_stale_firms(returns_df, year_Y, window=10, threshold=0.50):
    """
    Identifies firms with an excessive proportion of 0% returns 
    over the rolling window (e.g., 10 years) preceding year_Y.
    """
    start = year_Y - window + 1
    mask  = ((returns_df.columns.year >= start) &
             (returns_df.columns.year <= year_Y))
    window_ret = returns_df.loc[:, mask]

    if window_ret.empty:
        return pd.Index([])

    prop_zero = (window_ret == 0).sum(axis=1) / window_ret.shape[1]
    return prop_zero[prop_zero > threshold].index

In [3]:
# =====================================================================
# 2. INVESTMENT SET (ELIGIBILITY)
# =====================================================================

def get_eligible_firms(returns_df, prices_df, co2_df, rev_df, year_Y, min_obs=36):
    """
    Returns the eligible firms for investment at the end of December of year_Y.
    Criteria:
      1. Price is available and strictly positive at the end of Dec year_Y.
      2. At least min_obs months of returns available over the estimation window.
      3. Not stale (proportion of 0% returns <= 50%).
      4. CO2 and Revenue data available for year_Y.
    """
    # 1. Price available at the end of December year_Y
    dec_cols = prices_df.columns[prices_df.columns.year == year_Y]
    if len(dec_cols) == 0:
        return pd.Index([])
    dec_col = dec_cols[-1]
    has_price = prices_df[dec_col].dropna()
    has_price = has_price[has_price > 0].index

    # 2. Enough return observations over the last 10 years
    start = year_Y - 9
    mask  = ((returns_df.columns.year >= start) &
             (returns_df.columns.year <= year_Y))
    ret_window = returns_df.loc[:, mask]
    enough_obs = ret_window.notna().sum(axis=1)
    enough_obs = enough_obs[enough_obs >= min_obs].index

    # 3. Filter out stale firms
    stale = get_stale_firms(returns_df, year_Y)

    # 4. CO2 and Revenue data availability
    has_co2 = pd.Index([])
    if year_Y in co2_df.columns:
        has_co2 = co2_df[year_Y].dropna().index

    has_rev = pd.Index([])
    if year_Y in rev_df.columns:
        has_rev = rev_df[year_Y].dropna()
        has_rev = has_rev[has_rev > 0].index

    # Intersection of all eligibility criteria
    eligible = (has_price
                .intersection(enough_obs)
                .intersection(has_co2)
                .intersection(has_rev)
                .difference(stale))

    return eligible

In [4]:
# =====================================================================
# 3. COVARIANCE AND OPTIMIZATION
# =====================================================================

def estimate_sigma(returns_df, firms, year_Y, window=10):
    """
    Estimates the covariance matrix over the 10-year window 
    preceding year_Y for eligible firms.
    Includes diagonal regularization to ensure positive semi-definiteness.
    """
    start = year_Y - window + 1
    mask  = ((returns_df.columns.year >= start) &
             (returns_df.columns.year <= year_Y))
    R = returns_df.loc[firms, mask].fillna(0).values  # Shape: (N, T)
    Sigma = np.cov(R) + np.eye(len(firms)) * 1e-8
    return Sigma


def optimize_min_variance(Sigma):
    """
    Long-only minimum variance portfolio: min w'Σw  s.t.  sum(w)=1, w>=0.
    Returns the optimal weights as a numpy array.
    """
    N  = Sigma.shape[0]
    w0 = np.ones(N) / N

    result = minimize(
        lambda w: w @ Sigma @ w,
        w0,
        method='SLSQP',
        bounds=[(0, 1)] * N,
        constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
        options={'ftol': 1e-12, 'maxiter': 1000}
    )

    if not result.success:
        print("⚠️ Optimization failed to converge — applying equal weights.")
        return w0

    w = np.maximum(result.x, 0)
    return w / w.sum()

In [5]:
# =====================================================================
# 4. EX-POST PERFORMANCE (PORTFOLIO DRIFT)
# =====================================================================

def compute_yearly_perf(returns_df, firms, weights_arr, year_Y):
    """
    Calculates ex-post monthly returns for year_Y + 1.
    Weights are fixed at the end of Dec year_Y and drift naturally month by month.

    Drift formula: α_{i,t+1} = α_{i,t} * (1 + R_{i,t}) / (1 + R_{p,t})
    """
    dates = returns_df.columns[returns_df.columns.year == year_Y + 1]
    w     = weights_arr.copy()
    monthly = {}

    for k, date in enumerate(dates):
        r_vec = returns_df.loc[firms, date].fillna(0).values
        rp    = w @ r_vec
        monthly[date] = rp

        # Weight drift calculation for the next month
        if k < len(dates) - 1:
            w_new = w * (1 + r_vec)
            s     = w_new.sum()
            w     = w_new / s if s > 0 else w

    return pd.Series(monthly)

In [11]:
# =====================================================================
# 5. CHARGEMENT DES DONNÉES ET EXÉCUTION (CELLULE 6)
# =====================================================================
import os
import pandas as pd

INPUT_DIR = "../../../Data_cleaned" 

print("📥 Chargement des données CSV...")
prices_cl = pd.read_csv(os.path.join(INPUT_DIR, "Prices_clean.csv"), index_col=0)
co2_cl = pd.read_csv(os.path.join(INPUT_DIR, "CO2_clean.csv"), index_col=0)
rev_cl = pd.read_csv(os.path.join(INPUT_DIR, "Revenue_clean.csv"), index_col=0)

# --- CORRECTIONS DES TYPES DE COLONNES ---
# 1. On force les en-têtes de colonnes Prix à redevenir des "Dates"
prices_cl.columns = pd.to_datetime(prices_cl.columns)
# 2. On force les en-têtes CO2 et Revenus à redevenir des "Nombres Entiers"
co2_cl.columns = co2_cl.columns.astype(int)
rev_cl.columns = rev_cl.columns.astype(int)
# -----------------------------------------

# 2. Calcul des rendements mensuels
returns_m = compute_returns(prices_cl)

# 3. La boucle d'allocation Minimum Variance
weights_mv = {}
print("🚀 Lancement de l'allocation (2013-2024)...")

for year in range(2013, 2025):
    print(f"🔄 Calcul pour l'année {year}...")
    
    # A. Trouver les firmes valides
    firms = get_eligible_firms(returns_m, prices_cl, co2_cl, rev_cl, year)
    print(f"   -> {len(firms)} firmes éligibles trouvées.")
    
    if len(firms) > 0:
        # B. Calculer la matrice de covariance
        Sigma = estimate_sigma(returns_m, firms, year)
        
        # C. Trouver les poids optimaux
        w = optimize_min_variance(Sigma)
        
        # D. Stocker le résultat
        weights_mv[year] = pd.Series(w, index=firms)
    else:
        print(f"⚠️ Aucune firme éligible en {year}")

# 4. Export vers Excel
OUTPUT_DIR = INPUT_DIR
df_mv = pd.DataFrame(weights_mv).T
excel_path = os.path.join(OUTPUT_DIR, "SAAM_Base_Portfolio.xlsx")

with pd.ExcelWriter(excel_path) as writer:
    df_mv.to_excel(writer, sheet_name="Min_Variance")

print(f"✅ Fichier Excel généré avec succès ici : {excel_path}")

📥 Chargement des données CSV...


C:\Users\nourb\AppData\Local\Temp\ipykernel_18004\2107181166.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  returns = prices_df.pct_change(axis=1, fill_method=None)


🚀 Lancement de l'allocation (2013-2024)...
🔄 Calcul pour l'année 2013...
   -> 357 firmes éligibles trouvées.
🔄 Calcul pour l'année 2014...
   -> 365 firmes éligibles trouvées.
🔄 Calcul pour l'année 2015...
   -> 380 firmes éligibles trouvées.
🔄 Calcul pour l'année 2016...
   -> 396 firmes éligibles trouvées.
🔄 Calcul pour l'année 2017...
   -> 415 firmes éligibles trouvées.
🔄 Calcul pour l'année 2018...
   -> 452 firmes éligibles trouvées.
🔄 Calcul pour l'année 2019...
   -> 516 firmes éligibles trouvées.
🔄 Calcul pour l'année 2020...
   -> 541 firmes éligibles trouvées.
🔄 Calcul pour l'année 2021...
   -> 564 firmes éligibles trouvées.
🔄 Calcul pour l'année 2022...
   -> 574 firmes éligibles trouvées.
🔄 Calcul pour l'année 2023...
   -> 574 firmes éligibles trouvées.
🔄 Calcul pour l'année 2024...
   -> 571 firmes éligibles trouvées.
✅ Fichier Excel généré avec succès ici : ../../../Data_cleaned\SAAM_Base_Portfolio.xlsx


In [13]:
# =====================================================================
# 7. CALCUL DES RENDEMENTS ET CRÉATION DU RAPPORT FINAL COMPLET (CELLULE 7)
# =====================================================================
import numpy as np

print("📈 Calcul des performances mensuelles ex-post...")
all_monthly_returns = []

# On simule les rendements
for year in range(2013, 2024):
    if year in weights_mv:
        w = weights_mv[year].values
        firms = weights_mv[year].index
        perf_series = compute_yearly_perf(returns_m, firms, w, year)
        all_monthly_returns.append(perf_series)

# 1. On regroupe les rendements
mv_returns = pd.concat(all_monthly_returns)
mv_returns.name = "MV_Return"

# 2. On calcule le rendement cumulé (Base 100)
mv_cumul = 100 * (1 + mv_returns).cumprod()
mv_cumul.name = "MV_Cumul"

df_perf = pd.concat([mv_returns, mv_cumul], axis=1)

# ==========================================
# 3. NOUVEAU : CALCUL DES SUMMARY STATS
# ==========================================
print("📊 Calcul des statistiques de performance (Sharpe, Volatilité)...")

ann_return = mv_returns.mean() * 12 * 100
ann_vol = mv_returns.std() * np.sqrt(12) * 100
sharpe = ann_return / ann_vol  # Ratio de Sharpe simplifié
min_ret = mv_returns.min() * 100
max_ret = mv_returns.max() * 100
n_months = len(mv_returns)

summary_stats = pd.DataFrame({
    "Ann. Return (%)": [ann_return],
    "Ann. Volatility (%)": [ann_vol],
    "Sharpe Ratio": [sharpe],
    "Min Monthly Ret (%)": [min_ret],
    "Max Monthly Ret (%)": [max_ret],
    "N Months": [n_months]
}, index=["MV Portfolio P(mv)_oos"])

# ==========================================
# 4. EXPORT DU MEGA-EXCEL MULTI-ONGLETS
# ==========================================
final_excel_path = os.path.join(OUTPUT_DIR, "SAAM_Resultat_Final.xlsx")

with pd.ExcelWriter(final_excel_path) as writer:
    df_perf.to_excel(writer, sheet_name="Monthly_Returns")
    summary_stats.to_excel(writer, sheet_name="Summary_Stats") # Le nouvel onglet !
    df_mv.to_excel(writer, sheet_name="Weights_History")
    
    # Ajout de toutes tes données de base (comme sur ta photo)
    prices_cl.to_excel(writer, sheet_name="Prices")
    returns_m.to_excel(writer, sheet_name="Returns")
    co2_cl.to_excel(writer, sheet_name="CO2")
    rev_cl.to_excel(writer, sheet_name="Revenue")

print(f"🏆 Super-Rapport généré avec succès ici : {final_excel_path}")

📈 Calcul des performances mensuelles ex-post...
📊 Calcul des statistiques de performance (Sharpe, Volatilité)...


PermissionError: [Errno 13] Permission denied: '../../../Data_cleaned\\SAAM_Resultat_Final.xlsx'